### RapidFire AI Knowledge RAG experiment with RapidFire AI API and TritonAI API

In [1]:
# import getpass
# TRITON_API_KEY = getpass.getpass("Enter your Triton API key: ")

# The above option is probably safer
from pathlib import Path
TRITON_API_KEY = Path("~/Project1/api-key.txt").expanduser().read_text(encoding="utf-8").splitlines()[0].strip()

In [2]:
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)
###################### Import before Experiment
# from rapidfireai_datahub_compat import (
#     patch_rapidfireai_for_datahub
# )
# print(patch_rapidfireai_for_datahub())
######################
from rapidfireai import Experiment

import re, json, math
from typing import List as listtype, Dict, Any

# project1_eval.py must be in the same directory as this notebook.
# It is shared across evaluate_retrieval.py, run_judge.py, and this notebook;
# do not modify the file itself.
from eval_scripts.project1_eval import (
    to_spans,           # validates + coerces (file, start, end) tuples
    precision_at_k,     # overlap-aware Precision@k
    recall_at_k,        # overlap-aware Recall@k
    f1_at_k,            # harmonic mean of the above
    call_judge,         # LLM-as-judge: correctness / faithfulness / completeness
)


INFO 04-28 12:53:15 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 04-28 12:53:15 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.


##### ⚠️ API Cost Considerations
This notebook runs 4 configurations concurrently on a downsampled dataset of 256 examples.
Estimated Costs:
- Current run (downsampled): \$5 
- Full set: \$45

> 💡 **Tip:** Monitor your API usage to avoid unexpected charges.

### Load Queries to `qrels`

In [3]:
import pandas as pd
from datasets import Dataset
from pathlib import Path

# Load golden QA pairs from validation set
validation_file = Path("~/Project1/validation-set-golden-qa-pairs.json").expanduser()
with open(validation_file, "r") as f:
    qa_pairs = json.load(f)

# Build queries dataset.
# reference_answer is passed through the batch so postprocess_fn can read it
# without reloading the file on every batch.
queries = {
    "query":            [qa["question"]         for qa in qa_pairs],
    "query_id":         [str(qa["question_id"]) for qa in qa_pairs],
    "reference_answer": [qa["reference_answer"] for qa in qa_pairs],
}
queries = Dataset.from_dict(queries).shuffle(seed=42)

# Build qrels: one row per (query_id, evidence span).
# lines is normalised to a strict 2-element [start_line, end_line] list so it
# is always compatible with project1_eval.to_spans({file, lines: [s, e]}).
qrels_data = []
for qa in qa_pairs:
    query_id = str(qa["question_id"])
    for evidence in qa["source_evidence"]:
        raw_lines = evidence.get("lines", [])
        if len(raw_lines) >= 2:
            start_line, end_line = int(min(raw_lines)), int(max(raw_lines))
        elif len(raw_lines) == 1:
            start_line = end_line = int(raw_lines[0])
        else:
            # No line info — skip; a span with unknown bounds cannot be evaluated
            continue
        qrels_data.append({
            "query_id":   query_id,
            "file":       Path(evidence.get("file", "")).name,
            "start_line": start_line,
            "end_line":   end_line,
            "relevance":  1,
        })

qrels = pd.DataFrame(qrels_data)
print(f"Loaded {len(qa_pairs)} QA pairs")
print(f"Dataset shape: {queries.shape}")
print(f"Qrels shape:   {qrels.shape}")
qrels.head(10)


Loaded 45 QA pairs
Dataset shape: (45, 3)
Qrels shape:   (52, 5)


,query_id,file,start_line,end_line,relevance
0,1,configs.rst,23,48,1
1,2,difference.rst,4,29,1
2,3,experiment.rst,27,39,1
3,4,configs.rst,58,102,1
4,5,walkthroughrag.rst,29,103,1
5,6,icops.rst,75,176,1
6,7,evalsfunctions.rst,8,31,1
7,8,ragspecs.rst,56,98,1
8,8,rag_fiqa.rst,49,59,1
9,9,experiment.rst,215,219,1


### Create Experiment

In [4]:
# from rapidfireai import Experiment
experiment = Experiment(experiment_name="exp1", mode="evals")

An experiment with the same name already exists. Created a new experiment 'exp1_1' with Experiment ID: 2 at /home/yup031/rapidfireai/rapidfire_experiments/exp1_1
Created directory: /home/yup031/rapidfireai/logs/exp1_1


### Define Partial Multi-Config Knobs for LangChain part of RAG Pipeline using RapidFire AI Wrapper APIs

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from typing import Dict

batch_size = 32

def custom_template(retrieved_docs: listtype[Document]) -> str:
    """
    Custom prompt template that combines the content of retrieved documents into a single context string for the generator model.
    """
    formatted_docs = []
    for doc in retrieved_docs:
        content = doc.page_content
        formatted_doc = f"{content}"  # only include helpful content without metadata
        formatted_docs.append(formatted_doc)
    
    # Combine all formatted documents into a single context string
    context = "\n\n".join(formatted_docs)
    
    return context


# CPU-based RAG
rag_cpu = RFLangChainRagSpec(
    document_loader=DirectoryLoader(
        path="~/Project1/sourcedocs",
        glob="*.rst",
        loader_cls=JSONLoader,
        sample_seed=1337,
    ),
    text_splitter=List([
            RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name="gpt2", chunk_size=256, chunk_overlap=32
            ),
            RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name="gpt2", chunk_size=128, chunk_overlap=32
            ),
        ],
    ),
    embedding_cfg=List([
        # {
        #     "class": HuggingFaceEmbeddings,
        #     "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        #     "model_kwargs": {"device": "cpu"},
        #     "encode_kwargs": {"normalize_embeddings": True, "batch_size": batch_size},
        # },
        { # For quicker demo in class
            "class": OpenAIEmbeddings,
            "model": "api-tgpt-embeddings",
            "api_key": TRITON_API_KEY,
            "base_url": "https://tritonai-api.ucsd.edu",
            "check_embedding_ctx_length": False,
        },
    ]),
    # FAISS is an in-memory store and only works in create mode. 
    vector_store_cfg={
        "type": "faiss", 
        "batch_size": batch_size
    }, # if not set, uses FAISS by default
    search_cfg=List([{"type": "similarity", "k": 10}, {"type": "mmr", "k": 10}]), # 2 different search types
    reranker_cfg={
        "class": CrossEncoderReranker,
        "model_name": "cross-encoder/ms-marco-MiniLM-L6-v2",
        "model_kwargs": {"device": "cpu"},
        "top_n": 5,
    },
    enable_gpu_search=False,
    document_template=custom_template,
)

### Define Data Processing and Postprocessing Functions

In [ ]:
def openai_sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Retrieve context and build generator prompts.

    Adds to batch
    -------------
    retrieved_spans    list[list[tuple[str, int, int]]]
        Per query, rank-ordered (file, start_line, end_line) tuples.
        Each tuple is validated by project1_eval.to_spans before being stored.
    serialized_context list[str]
        The exact text string passed to the generator for each query.
    prompts            list[list[dict]]
        OpenAI-style message list for each query.
    """
    import tiktoken
    from pathlib import Path

    SYSTEM_INSTRUCTIONS = (
        "You are a helpful QA assistant. "
        "Answer the question based solely on the provided context. "
        "Be concise and accurate."
    )
    MAX_TOKENS = 2000

    # Batched retrieval → list[list[Document]]
    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)

    # Build (file, start_line, end_line) spans.
    # Requires the document splitter to populate doc.metadata["start_line"] and
    # doc.metadata["end_line"] with valid 1-based, non-negative line numbers.
    # Use the rst_structural_splitter (which sets these natively) or the
    # add_line_numbers post-processing transform with token-based splitters.
    raw_spans_per_query = [
        [
            (
                Path(doc.metadata.get("source", "")).name,
                int(doc.metadata["start_line"]),
                int(doc.metadata["end_line"]),
            )
            for doc in docs
        ]
        for docs in all_context
    ]

    # Validate every span through project1_eval.to_spans so any malformed
    # chunk (start > end, span > 500 lines, missing file) raises immediately
    # here rather than silently corrupting the eval later.
    retrieved_spans: listtype[listtype[tuple]] = [
        to_spans(spans) for spans in raw_spans_per_query
    ]

    serialized_context: listtype[str] = rag.serialize_documents(all_context)
    batch["query_id"] = [str(query_id) for query_id in batch["query_id"]]

    enc = tiktoken.get_encoding("gpt2")

    def truncate_to_tokens(text: str, max_tokens: int) -> str:
        tokens = enc.encode(text)
        return enc.decode(tokens[:max_tokens]) if len(tokens) > max_tokens else text

    prompts = []
    for question, context in zip(batch["query"], serialized_context):
        user_content = truncate_to_tokens(
            f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:",
            MAX_TOKENS,
        )
        prompts.append([
            {"role": "system", "content": SYSTEM_INSTRUCTIONS},
            {"role": "user",   "content": user_content},
        ])

    return {
        "prompts":            prompts,
        "retrieved_spans":    retrieved_spans,
        "serialized_context": serialized_context,
        **batch,
    }


def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Attach ground-truth span and answer fields after generation.

    Adds to batch
    -------------
    ground_truth_spans  list[list[tuple[str, int, int]]]
        Gold evidence spans from qrels, validated by to_spans.
    reference_answer    list[str]
        Gold free-text answer (already in batch from the dataset; kept here
        explicitly so compute_metrics_fn can rely on it by name).
    """
    from pathlib import Path

    ground_truth_spans: listtype[listtype[tuple]] = []

    for query_id in batch["query_id"]:
        rows = qrels[qrels["query_id"] == str(query_id)]
        raw = [
            (row["file"], int(row["start_line"]), int(row["end_line"]))
            for _, row in rows.iterrows()
        ]
        # to_spans deduplicates and validates; any bad row raises ValueError here
        ground_truth_spans.append(to_spans(raw))

    batch["ground_truth_spans"] = ground_truth_spans

    # reference_answer was loaded into the dataset in cell 6; it arrives in
    # the batch automatically.  We assert its presence so the error is clear
    # if the dataset was built without it.
    assert "reference_answer" in batch, (
        "reference_answer missing from batch — make sure it is included when "
        "building the queries Dataset in cell 6."
    )

    return batch


In [ ]:
# Cell removed: was a stale SciFact-specific duplicate of the functions above.


### Define Custom Eval Metrics Functions

In [ ]:
# All retrieval metrics delegate to project1_eval so scoring is identical to
# the TA's evaluate_retrieval.py and run_judge.py scripts.

JUDGE_MODEL    = "gpt-4o-mini"
JUDGE_BASE_URL = "https://tritonai-api.ucsd.edu"   # set to None to use OpenAI directly
K = 5                                               # Precision/Recall/F1 cutoff


def sample_compute_metrics_fn(batch: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    """Compute retrieval and generation metrics for one batch.

    Reads from standardised batch fields:
        retrieved_spans    list[list[tuple[str, int, int]]]
        ground_truth_spans list[list[tuple[str, int, int]]]
        serialized_context list[str]   – context passed to generator
        generated_text     list[str]   – raw generator output
        reference_answer   list[str]   – gold answer
        query              list[str]   – original question

    All retrieval metrics use overlap-aware matching from project1_eval.
    Generation metrics use call_judge (LLM-as-judge).
    """
    total_queries = len(batch["query"])

    precisions, recalls, f1s = [], [], []
    correctness_scores, faithfulness_scores, completeness_scores = [], [], []
    judge_failures = 0

    for pred_spans, gt_spans, context, gen_text, ref_ans, query in zip(
        batch["retrieved_spans"],
        batch["ground_truth_spans"],
        batch["serialized_context"],
        batch["generated_text"],
        batch["reference_answer"],
        batch["query"],
    ):
        # ── retrieval metrics (overlap-aware, from project1_eval) ──────────
        precisions.append(precision_at_k(pred_spans, gt_spans, k=K))
        recalls.append(recall_at_k(pred_spans,    gt_spans, k=K))
        f1s.append(f1_at_k(pred_spans,            gt_spans, k=K))

        # ── generation quality via LLM-as-judge ────────────────────────────
        judge = call_judge(
            query=query,
            reference_answer=ref_ans,
            retrieved_context=context,
            system_answer=gen_text,
            model=JUDGE_MODEL,
            base_url=JUDGE_BASE_URL,
            api_key=TRITON_API_KEY,
        )
        if judge.get("failed"):
            judge_failures += 1
        correctness_scores.append(judge["correctness"])
        faithfulness_scores.append(judge["faithfulness"])
        completeness_scores.append(judge["completeness"])

    def mean(xs): return sum(xs) / len(xs) if xs else 0.0

    return {
        "Total":        {"value": total_queries},
        # Retrieval
        f"Precision@{K}": {"value": mean(precisions)},
        f"Recall@{K}":    {"value": mean(recalls)},
        f"F1@{K}":        {"value": mean(f1s)},
        # Generation (judge)
        "Correctness":    {"value": mean(correctness_scores)},   # 0 or 1
        "Faithfulness":   {"value": mean(faithfulness_scores)},  # 0 or 1
        "Completeness":   {"value": mean(completeness_scores)},  # 0-5
        "JudgeFailures":  {"value": judge_failures},
    }


def sample_accumulate_metrics_fn(
    aggregated_metrics: Dict[str, listtype],
) -> Dict[str, Dict[str, Any]]:
    """Weighted average of per-batch metrics across all shards."""
    num_per_batch = [m["value"] for m in aggregated_metrics["Total"]]
    total = sum(num_per_batch)

    weighted_keys = [
        f"Precision@{K}", f"Recall@{K}", f"F1@{K}",
        "Correctness", "Faithfulness",
    ]
    # Completeness is 0-5, not 0-1, so we track its range separately
    completeness_range = (0, 5)
    summed_keys = ["JudgeFailures"]  # accumulated as a raw count, not averaged

    result = {"Total": {"value": total}}

    for key in weighted_keys:
        result[key] = {
            "value": sum(
                m["value"] * q
                for m, q in zip(aggregated_metrics[key], num_per_batch)
            ) / total,
            "is_algebraic": True,
            "value_range": (0, 1),
        }

    result["Completeness"] = {
        "value": sum(
            m["value"] * q
            for m, q in zip(aggregated_metrics["Completeness"], num_per_batch)
        ) / total,
        "is_algebraic": True,
        "value_range": completeness_range,
    }

    result["JudgeFailures"] = {
        "value": sum(m["value"] for m in aggregated_metrics["JudgeFailures"]),
        "is_algebraic": False,
    }

    return result


### Define Partial Multi-Config Knobs for OpenAI Generator part of RAG Pipeline using RapidFire AI Wrapper APIs

In [ ]:
# 2 generator configs with different sizes of generator models and different reasoning levels
openai_config = RFOpenAIAPIModelConfig(
    client_config={"api_key": TRITON_API_KEY, "base_url": "https://tritonai-api.ucsd.edu", "max_retries": 2},
    model_config={
        "model": "api-mistral-small-3.2-2506",
        "max_completion_tokens": 2048,
    },
    rpm_limit=120, # Request per minute (RPM) needs to be set based on your account tier and the specific model used
    tpm_limit=1_000_000, # Token per minute (TPM) needs to be set based on your account tier and the specific model used
    rag=rag_cpu,
    prompt_manager=None,
)


config_set = List([{
        "openai_config": openai_config,
        "batch_size": batch_size,
        "preprocess_fn": openai_sample_preprocess_fn,
        "postprocess_fn": sample_postprocess_fn,
        "compute_metrics_fn": sample_compute_metrics_fn,
        "accumulate_metrics_fn": sample_accumulate_metrics_fn,
        "online_strategy_kwargs": {
            "strategy_name": "normal",
            "confidence_level": 0.95,
            "use_fpc": True,
        },
    },
])

### Create Config Group

In [ ]:
# Simple grid search across all sets of config knob values = 4 combinations in total
config_group = RFGridSearch(config_set)

### Run Multi-Config Evals

In [ ]:
# Launch evals of all RAG configs in the config_group with swap granularity of 4 chunks
results = experiment.run_evals(
    config_group=config_group,
    dataset=queries,
    num_shards=4,
    seed=42,
)


### View Results

In [ ]:
# Convert results dict to DataFrame – one row per config, one column per metric
results_df = pd.DataFrame([
    {
        "run_id": run_id,
        **{
            k: (v["value"] if isinstance(v, dict) and "value" in v else v)
            for k, v in metrics_dict.items()
        },
    }
    for run_id, (_, metrics_dict) in results.items()
])

# Reorder columns: retrieval metrics first, then judge metrics
retrieval_cols = [c for c in results_df.columns if any(
    c.startswith(p) for p in ["Precision", "Recall", "F1"])]
judge_cols     = ["Correctness", "Faithfulness", "Completeness", "JudgeFailures"]
other_cols     = [c for c in results_df.columns
                  if c not in retrieval_cols + judge_cols + ["run_id", "Total"]]

results_df = results_df[
    ["run_id", "Total"] + retrieval_cols + judge_cols + other_cols
]
results_df


### End Experiment

In [ ]:
experiment.end()

### View RapidFire AI Log Files

In [ ]:
# Get the experiment-specific log file
log_file = experiment.get_log_file_path()

print(f"📄 Log File: {log_file}")
print()

if log_file.exists():
    print("=" * 80)
    print(f"Last 30 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())
else:
    print(f"❌ Log file not found: {log_file}")